#  ML Research Assistant - Advanced RAG System

> **AI-powered research companion** for exploring 5,000+ ArXiv ML/AI papers  
> Features: Citation-backed answers • Quality scoring • PDF upload 
---

###  What It Does
- Ask questions in natural language across academic literature
- Get accurate answers with source citations and quality metrics
- Upload your own papers to expand the knowledge base
- Export chat history for documentation


###  Quick Start

Ask questions or upload PDFs

---

**Built for researchers, students, and ML practitioners**

_____

# Import Libraries & Environment Check
Load all dependencies and verify GPU, FAISS, PyMuPDF, and Gradio are working.

In [2]:
import os, re, pickle
from pathlib import Path
from typing import List, Dict
import torch, faiss, fitz, gradio as gr
import nltk
import nltk
from datasets import load_dataset
from langchain_core.documents import Document
import torch, gc
import os, re, pickle, hashlib
import numpy as np
import torch
from typing import List, Dict, Tuple
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

import torch, pickle, re, traceback, os, csv, io, gc, shutil
import gradio as gr
import numpy as np
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import CrossEncoder
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Kill any existing processes
torch.cuda.empty_cache()
gc.collect()
# Set memory fraction
torch.cuda.set_per_process_memory_fraction(0.95, 0)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")
print(f"FAISS   : {faiss.__version__}")
print(f"PyMuPDF : fitz.open={hasattr(fitz,'open')}")
print(f"Gradio  : {gr.__version__}")
Path('/kaggle/working').mkdir(exist_ok=True)

GPU     : Tesla T4
FAISS   : 1.13.2
PyMuPDF : fitz.open=True
Gradio  : 5.50.0


# Load ArXiv Dataset

In [3]:
def load_arxiv_papers(max_papers: int = 5000, save_path : str = '/kaggle/working/raw_papers.pkl') -> List[Dict]:

    print(f" Loading up to {max_papers} papers from HuggingFace...")
    print("   Dataset: CShorten/ML-ArXiv-Papers (~118k ML papers)")
    # print("   No PDFs — abstracts only — fast and reliable\n")

    dataset = load_dataset(
        "CShorten/ML-ArXiv-Papers",
        split     = "train",
        streaming = True,
    )

    papers, count = [], 0

    for paper in dataset:
        abstract = (paper.get('abstract') or '').strip()
        title    = (paper.get('title')    or '').strip()

        if len(abstract) < 100 or not title:
            continue

        papers.append({
            'id'       : str(count),
            'title'    : title,
            'abstract' : abstract,
            'text'     : title + '\n\n' + abstract,
        })
        count += 1

        if count % 500 == 0:
            print(f"    {count}/{max_papers} collected...")

        if count >= max_papers:
            break

    print(f"\n Loaded {len(papers)} papers")
    print(f"   Avg abstract: {sum(len(p['abstract']) for p in papers)//len(papers)} chars")

    with open(save_path, 'wb') as f:
        pickle.dump(papers, f)

    return papers

#### Remove LaTeX equations, URLs, citations, and normalize special characters.

In [4]:
def clean_text(text: str) -> str:
    text = re.sub(r'\$\$.*?\$\$', '', text, flags=re.DOTALL)
    text = re.sub(r'\$.*?\$',     '', text)
    text = re.sub(r'\[\d+(?:[-,]\d+)*\]', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    for old, new in [('\u2019',"'"),('\u2018',"'"),
                     ('\u201c','"'),('\u201d','"'),
                     ('\u2013','-'),('\u2014',' - ')]:
        text = text.replace(old, new)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+',  ' ',   text)
    return text.strip()

#### Convert to LangChain Documents

In [5]:
def papers_to_docs(papers: List[Dict]) -> List[Document]:
    docs = []
    for p in papers:
        docs.append(Document(
            page_content = clean_text(p['text']),
            metadata = {
                'paper_id'  : p['id'],
                'title'     : p['title'],
                'source'    : (
                    "https://arxiv.org/search/?query="
                    + p['title'][:60].replace(' ', '+')
                    + "&searchtype=ti"
                ),
            }
        ))
    print(f" Created {len(docs)} LangChain Documents")
    return docs

### Execute data loading and document preparation.

In [6]:
papers    = load_arxiv_papers(max_papers=5000)
documents = papers_to_docs(papers)

with open('/kaggle/working/documents.pkl', 'wb') as f:
    pickle.dump(documents, f)

print(f"   Papers    : {len(papers)}")
print(f"   Documents : /kaggle/working/documents.pkl")

 Loading up to 5000 papers from HuggingFace...
   Dataset: CShorten/ML-ArXiv-Papers (~118k ML papers)


README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

    500/5000 collected...
    1000/5000 collected...
    1500/5000 collected...
    2000/5000 collected...
    2500/5000 collected...
    3000/5000 collected...
    3500/5000 collected...
    4000/5000 collected...
    4500/5000 collected...
    5000/5000 collected...

 Loaded 5000 papers
   Avg abstract: 1017 chars
 Created 5000 LangChain Documents
   Papers    : 5000
   Documents : /kaggle/working/documents.pkl


#### Semantic Chunking Class
Intelligent chunking that detects and preserves paper structure (Abstract, Methods, etc.).

#### Query Expansion Class
Expand user queries with technical synonyms and related ML/AI terms for better retrieval.

In [8]:
class QueryExpander:
    """Expand queries with technical synonyms and related terms"""
    
    def __init__(self):
        # Technical synonym mappings for ML/AI domain
        self.synonym_map = {
            'neural network': ['deep learning', 'artificial neural network', 'ANN', 'deep net'],
            'transformer': ['attention mechanism', 'self-attention', 'multi-head attention'],
            'bert': ['bidirectional encoder', 'masked language model', 'MLM'],
            'gpt': ['generative pre-training', 'autoregressive', 'causal language model'],
            'cnn': ['convolutional neural network', 'convnet', 'image classification'],
            'rnn': ['recurrent neural network', 'sequence model', 'LSTM', 'GRU'],
            'gradient descent': ['backpropagation', 'optimization', 'weight update'],
            'loss function': ['objective function', 'cost function', 'error metric'],
            'overfitting': ['generalization', 'regularization', 'model capacity'],
            'embedding': ['representation learning', 'vector representation', 'latent space'],
            'fine-tuning': ['transfer learning', 'adaptation', 'downstream task'],
            'attention': ['self-attention', 'cross-attention', 'attention weights'],
            'rl': ['reinforcement learning', 'policy gradient', 'q-learning'],
        }
    
    def expand(self, query: str, max_expansions: int = 2) -> List[str]:
        """Generate query variations"""
        expansions = [query]  # Always include original
        query_lower = query.lower()
        
        # Find matching terms and add synonyms
        for term, synonyms in self.synonym_map.items():
            if term in query_lower:
                # Add up to max_expansions synonyms
                for syn in synonyms[:max_expansions]:
                    # Replace term with synonym in query
                    expanded = re.sub(
                        re.escape(term), 
                        syn, 
                        query_lower, 
                        flags=re.IGNORECASE
                    )
                    if expanded not in expansions:
                        expansions.append(expanded)
        
        return expansions[:3]  # Limit to 3 total variations




Combine vector search and BM25 keyword search results with configurable weights.

In [10]:
def weighted_rrf(vector_results: List,bm25_results: List[Dict],vector_weight: float = 0.6,
    bm25_weight: float = 0.4,
    k: int = 60,
    top_k: int = 20
) -> List[Dict]:
    """
    Weighted Reciprocal Rank Fusion
    Gives more control over vector vs keyword search importance
    """
    scores, metadata = {}, {}
    
    # Process vector results with weight
    for rank, doc in enumerate(vector_results):
        text = doc.page_content
        scores[text] = scores.get(text, 0) + vector_weight / (k + rank + 1)
        metadata[text] = doc.metadata
    
    # Process BM25 results with weight
    for rank, doc in enumerate(bm25_results):
        text = doc['text']
        scores[text] = scores.get(text, 0) + bm25_weight / (k + rank + 1)
        if text not in metadata:
            metadata[text] = doc.get('metadata', {})
    
    # Sort by combined score
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    return [
        {
            'text': t, 
            'score': round(s, 6), 
            'metadata': metadata.get(t, {}),
            'fusion_method': 'weighted_rrf'
        }
        for t, s in sorted_items[:top_k]
    ]



Apply Maximal Marginal Relevance to diversify results and avoid redundant chunks.

In [11]:
def apply_mmr(
    candidates: List[Dict],
    query_embedding: np.ndarray,
    embed_model,
    lambda_param: float = 0.5,
    top_k: int = 5
) -> List[Dict]:
    """
    Diversify results to avoid returning multiple chunks from same paper
    lambda_param: 0 = max diversity, 1 = max relevance
    """
    if len(candidates) <= top_k:
        return candidates
    
    selected = []
    remaining = candidates.copy()
    
    # Pre-compute all embeddings (batch for efficiency)
    remaining_texts = [c['text'][:500] for c in remaining]  # Truncate for speed
    remaining_embeddings = np.array([
        embed_model.embed_query(text) for text in remaining_texts
    ])
    
    while len(selected) < top_k and remaining:
        if not selected:
            # First item: most relevant to query
            similarities = cosine_similarity([query_embedding], remaining_embeddings)[0]
            best_idx = int(np.argmax(similarities))
        else:
            # Balance relevance vs diversity
            query_sims = cosine_similarity([query_embedding], remaining_embeddings)[0]
            
            # Get embeddings of selected items
            selected_texts = [s['text'][:500] for s in selected]
            selected_embeddings = np.array([
                embed_model.embed_query(text) for text in selected_texts
            ])
            
            # Max similarity to any selected item
            max_similarities = cosine_similarity(
                remaining_embeddings, 
                selected_embeddings
            ).max(axis=1)
            
            # MMR score = λ * relevance - (1-λ) * similarity_to_selected
            mmr_scores = lambda_param * query_sims - (1 - lambda_param) * max_similarities
            best_idx = int(np.argmax(mmr_scores))
        
        # Move from remaining to selected
        selected.append(remaining[best_idx])
        del remaining[best_idx]
        remaining_embeddings = np.delete(remaining_embeddings, best_idx, axis=0)
    
    return selected



Traditional keyword-based search index for hybrid retrieval.

In [13]:

class BM25Index:
    def __init__(self):
        self.bm25   = None
        self.chunks = []

    def build(self, chunks):
        print(f"🔨 Building BM25 on {len(chunks):,} chunks...")
        self.chunks   = chunks
        tokenized     = [c.page_content.lower().split() for c in chunks]
        self.bm25     = BM25Okapi(tokenized)
        print(" BM25 ready")

    def search(self, query: str, top_k: int = 15) -> List[Dict]:
        if not self.bm25:
            raise ValueError("Call build() first")
        scores      = self.bm25.get_scores(query.lower().split())
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            if scores[idx] <= 0:
                break
            chunk = self.chunks[idx]
            results.append({
                'text'    : chunk.page_content,
                'score'   : float(scores[idx]),
                'metadata': chunk.metadata,
            })
        return results



Build vector store with BGE-large, BM25 index, and initialize all retrieval components.

In [15]:
GPU = torch.cuda.is_available()
NUM_GPUS = torch.cuda.device_count() if GPU else 0

if GPU:
    torch.cuda.empty_cache()
    gc.collect()

# Create uploads directory
UPLOADS_DIR = Path("/kaggle/working/uploaded_pdfs")
UPLOADS_DIR.mkdir(exist_ok=True)


def load_enhanced_phase2():
    embed_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5",
        model_kwargs={'device': 'cuda:0' if GPU else 'cpu'},
        encode_kwargs={'normalize_embeddings': True, 'batch_size': 32},
    )
    
    vector_store = FAISS.load_local(
        "/kaggle/working/faiss_index_enhanced",
        embed_model,
        allow_dangerous_deserialization=True,
    )
    
    with open('/kaggle/working/phase2_enhanced_output.pkl', 'rb') as f:
        data = pickle.load(f)
    
    return (vector_store, data['bm25'], data['chunks'], embed_model,
            data['query_expander'], data['semantic_cache'], data['ragas_eval'])


Load BGE reranker and Mistral-7B with dual GPU support and 4-bit quantization.

In [16]:
class Reranker:
    def __init__(self):
        self.model = CrossEncoder("BAAI/bge-reranker-large", 
                                  device='cuda:0' if GPU else 'cpu')
    
    def rerank(self, query: str, chunks: List[Dict], top_k: int = 5) -> List[Dict]:
        if not chunks:
            return []
        pairs = [(query, c['text'][:500]) for c in chunks]
        scores = self.model.predict(pairs, batch_size=16, show_progress_bar=False)
        for chunk, score in zip(chunks, scores):
            chunk['rerank_score'] = float(score)
        return sorted(chunks, key=lambda x: x['rerank_score'], reverse=True)[:top_k]


class Generator:
    MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
    
    def __init__(self):
        if GPU:
            torch.cuda.empty_cache()
        
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_ID)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model = AutoModelForCausalLM.from_pretrained(
            self.MODEL_ID, quantization_config=bnb,
            device_map="balanced" if NUM_GPUS >= 2 else "auto",
            max_memory={0: "14GB", 1: "14GB"} if NUM_GPUS >= 2 else {0: "13GB"},
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
        )
        self.model.eval()
    
    def build_prompt(self, query, chunks, history):
        system = "You are an AI research assistant. Answer only from the context. Cite sources as [Source N]."
        ctx = "\n".join([f"[Source {i+1}] {c['metadata'].get('title', 'Unknown')[:60]}\n{c['text']}" 
                        for i, c in enumerate(chunks)])
        hist = "".join([f"[INST] {u} [/INST] {b} </s> " for u, b in history[-3:]])
        return f"<s>[INST] {system}\n\nCONTEXT:\n{ctx} [/INST] Understood. </s>\n{hist}[INST] {query} [/INST]"
    
    def generate(self, prompt, temperature=0.1):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(self.model.device)
        if GPU:
            torch.cuda.empty_cache()
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=512, temperature=max(temperature, 0.01),
                do_sample=temperature > 0.05, repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        result = self.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        del inputs, out
        if GPU:
            torch.cuda.empty_cache()
        return result



Track Q&A sessions with timestamps, quality scores, and CSV export.

In [18]:

class ChatHistory:
    def __init__(self):
        self.history = []
    
    def add(self, q, a, score):
        self.history.append({
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'question': q, 'answer': a, 'quality': score
        })
    
    def save_csv(self):
        path = '/kaggle/working/chat_history.csv'
        with open(path, 'w') as f:
            writer = csv.DictWriter(f, fieldnames=['timestamp', 'question', 'answer', 'quality'])
            writer.writeheader()
            writer.writerows(self.history)
        return path
    
    def get_stats(self):
        if not self.history:
            return "<p style='color:#9ca3af;'>Ask a question to see stats</p>"
        
        avg = sum(h['quality'] for h in self.history) / len(self.history)
        return f"""
        <div style='background:linear-gradient(135deg,#667eea,#764ba2);padding:20px;border-radius:12px;color:white;'>
            <div style='font-size:36px;font-weight:700;'>{len(self.history)}</div>
            <div style='font-size:14px;opacity:0.9;'>Questions Asked</div>
            <div style='margin-top:16px;padding-top:16px;border-top:1px solid rgba(255,255,255,0.3);'>
                <div style='font-size:12px;opacity:0.8;'>Average Quality</div>
                <div style='font-size:24px;font-weight:600;'>{avg:.2f}/1.0</div>
            </div>
        </div>
        """
    
    def clear(self):
        self.history = []



Load models, create instances, and prepare system for launch.

In [20]:
def chat(query, history, temp):
    if not query.strip():
        return history, "<p style='color:#9ca3af;padding:40px;text-align:center;'>Ask a question!</p>"
    
    try:
        if GPU:
            torch.cuda.empty_cache()
        
        q_emb = emb.embed_query(query)
        hit, cached = cache.get(query, q_emb)
        
        if hit:
            best = cached
        else:
            vec = vs.similarity_search(query, k=15)
            bm = bm25.search(query, top_k=15)
            cands = weighted_rrf(vec, bm, top=20)
            if not cands:
                return history + [(query, "No relevant papers found")], "<p style='color:#ef4444;'>No results</p>"
            div = apply_mmr(cands, q_emb, emb, top=10)
            best = reranker.rerank(query, div, top_k=5)
            cache.set(query, q_emb, best)
        
        prompt = gen.build_prompt(query, best, history)
        ans = gen.generate(prompt, temp)
        
        if GPU:
            torch.cuda.empty_cache()
            gc.collect()
        
        scores = ragas.evaluate(query, ans, best)
        hist.add(query, ans, scores['overall'])
        
        badge, color = ("🟢 Excellent", "#10b981") if scores['overall'] >= 0.75 else \
                       ("🟡 Good", "#f59e0b") if scores['overall'] >= 0.6 else \
                       ("🟠 Fair", "#ef4444")
        
        n_up = sum(1 for c in best if c['metadata'].get('uploaded'))
        
        html = f"""
        <div style='font-family:system-ui;'>
          <div style='background:{color};color:white;padding:16px;border-radius:10px;text-align:center;margin-bottom:20px;'>
            <div style='font-size:24px;font-weight:700;'>{badge}</div>
            <div style='font-size:12px;opacity:0.85;margin-top:4px;'>Answer Quality: {scores['overall']:.2f}/1.0</div>
          </div>
          <div style='background:#f9fafb;padding:14px;border-radius:8px;margin-bottom:20px;text-align:center;font-size:14px;'>
            📚 Based on <strong>{len(best)}</strong> research paper(s)
            {f" · <strong>{n_up}</strong> uploaded by you" if n_up > 0 else ""}
          </div>
          <div style='font-size:15px;font-weight:600;color:#1f2937;margin-bottom:14px;border-bottom:2px solid #e5e7eb;padding-bottom:8px;'>
            📖 Sources Referenced
          </div>
        """
        
        for i, chunk in enumerate(best, 1):
            title = chunk['metadata'].get('title', 'Unknown')[:75]
            uploaded = chunk['metadata'].get('uploaded', False)
            upload_date = chunk['metadata'].get('upload_date', '')
            pdf_path = chunk['metadata'].get('pdf_path', '')
            preview = chunk['text'][:160].replace('\n', ' ') + "..."
            
            search_title = title.replace(' ', '+')
            arxiv_url = f"https://arxiv.org/search/?query={search_title}&searchtype=title"
            
            bg = "#fff7ed" if uploaded else "#eff6ff"
            border = "#f97316" if uploaded else "#3b82f6"
            tag = "📤 YOUR UPLOAD" if uploaded else "📄 ArXiv"
            tag_bg = "#f97316" if uploaded else "#3b82f6"
            
            html += f"""
            <div style='background:{bg};border-left:4px solid {border};padding:16px;margin-bottom:14px;border-radius:0 10px 10px 0;'>
              <div style='display:flex;align-items:center;gap:10px;margin-bottom:10px;'>
                <span style='background:{tag_bg};color:white;padding:4px 10px;border-radius:5px;font-size:11px;font-weight:600;white-space:nowrap;'>{tag}</span>
                <span style='font-weight:600;font-size:14px;color:#1f2937;flex:1;'>[{i}] {title}</span>
              </div>
              {f'<div style="font-size:11px;color:#6b7280;margin-bottom:8px;">📅 Uploaded: {upload_date}</div>' if uploaded and upload_date else ''}
              {f'<div style="font-size:11px;color:#6b7280;margin-bottom:8px;">📁 File: {Path(pdf_path).name}</div>' if uploaded and pdf_path else ''}
              <div style='font-size:12px;color:#6b7280;background:white;padding:12px;border-radius:6px;line-height:1.6;'>
                "{preview}"
              </div>
              {f'<div style="margin-top:10px;"><a href="{arxiv_url}" target="_blank" style="color:#3b82f6;text-decoration:none;font-size:12px;font-weight:500;">🔗 Search on ArXiv ↗</a></div>' if not uploaded else ''}
            </div>
            """
        
        html += "</div>"
        
        return history + [(query, ans)], html
    
    except Exception as e:
        print(traceback.format_exc())
        return history + [(query, f"Error: {str(e)}")], "<p style='color:#ef4444;'>Error occurred</p>"


def clear():
    hist.clear()
    if GPU:
        torch.cuda.empty_cache()
        gc.collect()
    return [], "<p style='color:#9ca3af;padding:40px;text-align:center;'>Chat cleared</p>"


css = """
.header {background:linear-gradient(135deg,#667eea,#764ba2);padding:28px;border-radius:12px;margin-bottom:24px;color:white;}
.title {font-size:36px;font-weight:700;margin-bottom:8px;}
.subtitle {font-size:17px;opacity:0.92;}
"""

with gr.Blocks(title="AI Research Assistant", theme=gr.themes.Soft(), css=css) as demo:
    
    gr.HTML("""
    <div class="header">
        <div class="title"> AI Research Assistant</div>
        <div class="subtitle">Ask questions across 5,000+ ML/AI research papers</div>
    </div>
    """)
    
    with gr.Tabs():
        with gr.Tab("💬 Chat"):
            with gr.Row():
                with gr.Column(scale=2):
                    chatbot = gr.Chatbot(height=550, show_label=False, type="tuples")
                    with gr.Row():
                        msg = gr.Textbox(placeholder="Ask anything about ML/AI research...", 
                                        show_label=False, scale=4, container=False)
                        send = gr.Button("Send", variant="primary", scale=1)
                    
                    gr.Examples([
                        ["How does self-attention work in Transformers?"],
                        ["What is transfer learning?"],
                        ["Explain gradient descent optimization"],
                        ["Compare BERT and GPT architectures"],
                    ], inputs=[msg])
                    
                    with gr.Row():
                        temp = gr.Slider(0, 1, 0.1, step=0.05, label="Creativity Level")
                        clr = gr.Button("Clear", size="sm")
                
                with gr.Column(scale=1):
                    gr.Markdown("### 📊 Answer Quality")
                    sources = gr.HTML("<p style='color:#9ca3af;padding:30px;text-align:center;'>Ask a question to see sources</p>")
                    gr.Markdown("### 📈 Your Session")
                    stats = gr.HTML(hist.get_stats())
                    dl = gr.Button("💾 Download History", size="sm")
                    dlf = gr.File(visible=False)
            
            send.click(chat, [msg, chatbot, temp], [chatbot, sources]
                      ).then(lambda: "", None, msg
                      ).then(hist.get_stats, None, stats)
            
            msg.submit(chat, [msg, chatbot, temp], [chatbot, sources]
                      ).then(lambda: "", None, msg
                      ).then(hist.get_stats, None, stats)
            
            clr.click(clear, None, [chatbot, sources]
                     ).then(hist.get_stats, None, stats)
            
            dl.click(hist.save_csv, None, dlf
                    ).then(lambda: gr.update(visible=True), None, dlf)
        
        with gr.Tab("📤 Upload Papers"):
            gr.Markdown("### Add Your Research Papers\nUpload PDFs to include them in searches alongside the 5,000 existing papers.")
            
            with gr.Row():
                with gr.Column():
                    pdfs = gr.File(label="Select PDF Files", file_types=[".pdf"], 
                                  file_count="multiple", type="filepath")
                    titles = gr.Textbox(label="Paper Titles (optional)", 
                                       placeholder="Comma-separated titles", lines=2)
                    up_btn = gr.Button("🚀 Index Papers", variant="primary", size="lg")
                
                with gr.Column():
                    status = gr.Textbox(label="Status", lines=12, 
                                       value="No papers uploaded yet", interactive=False)
            
            up_btn.click(pdf_idx.index_multiple, [pdfs, titles], status)
        
        with gr.Tab("📋 History"):
            gr.Markdown("### Your Research Session\nView and download your Q&A history")
            
            with gr.Row():
                with gr.Column():
                    ref = gr.Button("🔄 Refresh Stats")
                    h_stats = gr.HTML(hist.get_stats())
                
                with gr.Column():
                    h_dl = gr.Button("⬇️ Download CSV", variant="primary", size="lg")
                    h_file = gr.File()
            
            ref.click(hist.get_stats, None, h_stats)
            h_dl.click(hist.save_csv, None, h_file)
    
    gr.HTML("""
    <div style='text-align:center;padding:24px;color:#9ca3af;font-size:13px;'>
        <p>"RAG-powered AI • 5,000+ ML/AI research papers"</p>
    </div>
    """)

demo.launch(share=True, show_error=True)

/tmp/ipykernel_57/2735251159.py:108: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Research Assistant", theme=gr.themes.Soft(), css=css) as demo:
/tmp/ipykernel_57/2735251159.py:108: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="AI Research Assistant", theme=gr.themes.Soft(), css=css) as demo:
/tmp/ipykernel_57/2735251159.py:121: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=550, show_label=False, type="tuples")
/tmp/ipykernel_57/2735251159.py:121: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a9592ad8b51163b101.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
